# 04 · Approach 3: Full Hybrid (Dense + Sparse + FTS)

**Strategy:** Combine all three Pinecone search signals in a single namespace:
- **Dense vector** — LLM caption embedded with `llama-text-embed-v2` (semantic meaning)
- **Sparse vector** — dialogue embedded with `pinecone-sparse-english-v0` (keyword relevance)
- **FTS field** — raw dialogue text indexed with BM25 (exact phrase matching)

**Namespace:** `ns-hybrid`

**Blog post insight:** A single Pinecone index can store all three signal types together. The key design constraint: each query picks *one* ranking signal — but you can chain them: use FTS/sparse filters to narrow the candidate set, then dense vectors re-rank for semantic relevance. This gives you the precision of keyword search with the recall of semantic search.

---

## Signal comparison

| Signal | Strengths | Weaknesses |
|---|---|---|
| Dense (LLM caption) | Conceptual themes, mood, setting | Misses specific names/words |
| Sparse (dialogue BM25) | Exact character names, sound effects (KAPOW!) | No semantic understanding |
| FTS (BM25 w/ stemming) | Phrase matching, stemming ("run" → "running") | Still keyword-based |
| Filter + dense rerank | Precision of keywords + recall of semantics | Requires both signals |


In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone
from tqdm import tqdm

load_dotenv()

TRANSCRIPTS_DIR = Path("data/transcripts")
INDEX_NAME      = "comics-multimodal"
NAMESPACE       = "ns-hybrid"
BATCH_SIZE      = 50

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

info = pc.preview.indexes.describe(name=INDEX_NAME)
INDEX_HOST = info.host
index = pc.index(host=INDEX_HOST)
print(f"Connected to {INDEX_NAME}")

## Generate sparse vectors

We use Pinecone's `pinecone-sparse-english-v0` model (via the inference API) to embed dialogue text into sparse vectors. These complement the dense LLM caption vectors.

In [ ]:
records = [json.loads(p.read_text()) for p in sorted(TRANSCRIPTS_DIR.glob("*.json"))]
print(f"Loaded {len(records)} transcripts")

In [ ]:
def embed_sparse_batch(texts: list[str]) -> list[dict]:
    """Embed texts with pinecone-sparse-english-v0. Returns list of {indices, values} dicts."""
    result = pc.inference.embed(
        model="pinecone-sparse-english-v0",
        inputs=texts,
        parameters={"input_type": "passage"},
    )
    return [{"indices": e.sparse_indices, "values": e.sparse_values} for e in result]


# Cache sparse vectors in transcript JSON to avoid re-billing
for i in tqdm(range(0, len(records), BATCH_SIZE), desc="Sparse embedding dialogue"):
    batch = records[i : i + BATCH_SIZE]
    needs = [r for r in batch if not r.get("sparse_vector")]
    if not needs:
        continue
    texts = [r["all_dialogue"] or r["scene_descriptions"] for r in needs]
    sparse_vecs = embed_sparse_batch(texts)
    for r, sv in zip(needs, sparse_vecs):
        r["sparse_vector"] = sv
        path = TRANSCRIPTS_DIR / f"{r['page_id']}.json"
        path.write_text(json.dumps(r, ensure_ascii=False, indent=2))

print("Sparse vectors ready.")

## Upsert to Pinecone (ns-hybrid)

Each document carries:
- `embedding` — the LLM caption text (Pinecone auto-embeds with `llama-text-embed-v2`)
- `sparse_embedding` — pre-computed sparse vector from `pinecone-sparse-english-v0`
- `dialogue` — raw text for the FTS field

In [ ]:
documents = [
    {
        "_id":              r["page_id"],
        "embedding":        r["llm_caption"],        # integrated inference → dense vector
        "sparse_embedding": r["sparse_vector"],       # BYOV sparse vector
        "dialogue":         r["all_dialogue"],        # FTS field
        "comic_title":      r["comic_title"],
        "page_id":          r["page_id"],
        "page_num":         r["page_num"],
        "file_path":        r["file_path"],
        "llm_caption":      r["llm_caption"],
    }
    for r in records
]

index.documents.batch_upsert(
    namespace=NAMESPACE,
    documents=documents,
    batch_size=BATCH_SIZE,
    show_progress=True,
)
print(f"Upserted {len(documents)} pages to {NAMESPACE}.")

## Search demos — comparing all four query modes

In [ ]:
from IPython.display import display
from PIL import Image


def show_results(matches, label=""):
    if label:
        print(f"\n{'='*60}\n{label}\n{'='*60}")
    for m in matches:
        caption = getattr(m, "llm_caption", "") or ""
        print(f"[{m._score:.3f}] {m._id} — {caption[:100]}")
        fp = getattr(m, "file_path", None)
        if fp and Path(fp).exists():
            img = Image.open(fp)
            img.thumbnail((250, 350))
            display(img)

In [ ]:
# 1. Dense-only: semantic search on LLM captions
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": "a moment of betrayal and deception"}],
    include_fields=["page_id", "comic_title", "llm_caption", "file_path"],
)
show_results(resp.matches, "Dense only: 'a moment of betrayal and deception'")

In [ ]:
# 2. Sparse-only: keyword search on dialogue
sparse_query = pc.inference.embed(
    model="pinecone-sparse-english-v0",
    inputs=["KAPOW detective badge"],
    parameters={"input_type": "query"},
)[0]

resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{
        "type": "sparse_vector",
        "field": "sparse_embedding",
        "query": {"indices": sparse_query.sparse_indices, "values": sparse_query.sparse_values},
    }],
    include_fields=["page_id", "comic_title", "dialogue", "file_path"],
)
show_results(resp.matches, "Sparse only: 'KAPOW detective badge'")

In [ ]:
# 3. FTS: BM25 on dialogue field (stemming catches run/running/ran)
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "text", "fields": ["dialogue"], "query": "running from danger"}],
    include_fields=["page_id", "comic_title", "dialogue", "file_path"],
)
show_results(resp.matches, "FTS (BM25 + stemming): 'running from danger'")

In [ ]:
# 4. Filter + dense rerank — the "hybrid" pattern
# Keyword filter narrows candidates, dense vector re-ranks by semantic relevance
resp = index.documents.search(
    namespace=NAMESPACE,
    top_k=3,
    score_by=[{"type": "dense_vector", "field": "embedding", "query": "tense face-off between hero and villain"}],
    filter={"dialogue": {"$match_any": ["stop", "never", "finish"]}},
    include_fields=["page_id", "comic_title", "llm_caption", "file_path"],
)
show_results(resp.matches, "Keyword filter + dense rerank: semantic 'tense face-off' where dialogue has stop/never/finish")